# 05 · Collector monitor

Coverage of TomTom / HERE / bus snapshots synced to Drive by the GitHub Actions collector. Gaps here mean a failing workflow (check Actions and `docs/setup_rclone.md`).

In [1]:
# Setup: works on Colab (Drive + private repo) and locally (repo checkout).
import sys, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    os.environ["BHTRAFFIC_DATA_ROOT"] = "/content/drive/MyDrive/transito_bh"
    token = userdata.get("GH_TOKEN")  # Colab Secrets: fine-grained token, read-only on the repo
    # Clone (not pip-from-git): the package resolves data/ and paper/ relative to the repo root.
    url = f"https://{token}@github.com/Dias0202/Transito_bh.git"
    if os.path.isdir("/content/Transito_bh"):
        !git -C /content/Transito_bh pull -q {url} main
    else:
        !git clone -q {url} /content/Transito_bh
        !git -C /content/Transito_bh remote set-url origin https://github.com/Dias0202/Transito_bh.git
    %pip -q install -e "/content/Transito_bh[geo,viz]"
    sys.path.insert(0, "/content/Transito_bh/src")
    os.chdir("/content/Transito_bh/notebooks")
else:
    from pathlib import Path
    sys.path.insert(0, str(Path.cwd().parent / "src"))

In [2]:
from bhtraffic.ingest import collector
cov = collector.coverage()
cov.pivot(index='date', columns='source', values='snapshots') if len(cov) else 'no snapshots yet'

'no snapshots yet'

In [3]:
paths = collector.build()
paths

{}

In [4]:
import pandas as pd
from bhtraffic.config import processed_dir
tt = processed_dir('collector') / 'tomtom.parquet'
if tt.exists():
    df = pd.read_parquet(tt)
    df['hour'] = df.ts_brt.dt.hour
    display(df.pivot_table(index='corridor_id', columns='hour', values='delay_ratio', aggfunc='median').round(2))